In [2]:
import pandas as pd
import numpy as np

# Step 1: Load the core datasets
pipeline_df = pd.read_csv('../dataset/raw/sales_pipeline.csv')
teams_df = pd.read_csv('../dataset/raw/sales_teams.csv')
accounts_df = pd.read_csv('../dataset/raw/accounts.csv')

# Step 2: Inspect the main Pipeline table
print("PIPELINE DATASET INFO")
pipeline_df.info()

PIPELINE DATASET INFO
<class 'pandas.DataFrame'>
RangeIndex: 8800 entries, 0 to 8799
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   opportunity_id  8800 non-null   str    
 1   sales_agent     8800 non-null   str    
 2   product         8800 non-null   str    
 3   account         7375 non-null   str    
 4   deal_stage      8800 non-null   str    
 5   engage_date     8300 non-null   str    
 6   close_date      6711 non-null   str    
 7   close_value     6711 non-null   float64
dtypes: float64(1), str(7)
memory usage: 550.1 KB


In [6]:
from IPython.display import display

print("=== PIPELINE DATASET HEAD ===")
display(pipeline_df.head(10))

print("\n=== SALES TEAMS DATASET HEAD ===")
display(teams_df.head(10))

print("\n=== ACCOUNTS DATASET HEAD ===")
display(accounts_df.head(10))

=== PIPELINE DATASET HEAD ===


,opportunity_id,sales_agent,product,account,deal_stage,engage_date,close_date,close_value
0,1C1I7A6R,Moses Frase,GTX Plus Basic,Cancity,Won,2016-10-20,2017-03-01,1054.0
1,Z063OYW0,Darcel Schlecht,GTXPro,Isdom,Won,2016-10-25,2017-03-11,4514.0
2,EC4QE1BX,Darcel Schlecht,MG Special,Cancity,Won,2016-10-25,2017-03-07,50.0
3,MV1LWRNH,Moses Frase,GTX Basic,Codehow,Won,2016-10-25,2017-03-09,588.0
4,PE84CX4O,Zane Levy,GTX Basic,Hatfan,Won,2016-10-25,2017-03-02,517.0
5,ZNBS69V1,Anna Snelling,MG Special,Ron-tech,Won,2016-10-29,2017-03-01,49.0
6,9ME3374G,Vicki Laflamme,MG Special,J-Texon,Won,2016-10-30,2017-03-02,57.0
7,7GN8Q4LL,Markita Hansen,GTX Basic,Cheers,Won,2016-11-01,2017-03-07,601.0
8,OLK9LKZB,Niesha Huffines,GTX Plus Basic,Zumgoity,Won,2016-11-01,2017-03-03,1026.0
9,HAXMC4IX,James Ascencio,MG Advanced,NaN,Engaging,2016-11-03,NaN,NaN



=== SALES TEAMS DATASET HEAD ===


,sales_agent,manager,regional_office
0,Anna Snelling,Dustin Brinkmann,Central
1,Cecily Lampkin,Dustin Brinkmann,Central
2,Versie Hillebrand,Dustin Brinkmann,Central
3,Lajuana Vencill,Dustin Brinkmann,Central
4,Moses Frase,Dustin Brinkmann,Central
5,Jonathan Berthelot,Melvin Marxen,Central
6,Marty Freudenburg,Melvin Marxen,Central
7,Gladys Colclough,Melvin Marxen,Central
8,Niesha Huffines,Melvin Marxen,Central
9,Darcel Schlecht,Melvin Marxen,Central



=== ACCOUNTS DATASET HEAD ===


,account,sector,year_established,revenue,employees,office_location,subsidiary_of
0,Acme Corporation,technolgy,1996,1100.04,2822,United States,NaN
1,Betasoloin,medical,1999,251.41,495,United States,NaN
2,Betatech,medical,1986,647.18,1185,Kenya,NaN
3,Bioholding,medical,2012,587.34,1356,Philipines,NaN
4,Bioplex,medical,1991,326.82,1016,United States,NaN
5,Blackzim,retail,2009,497.11,1588,United States,NaN
6,Bluth Company,technolgy,1993,1242.32,3027,United States,Acme Corporation
7,Bubba Gump,software,2002,987.39,2253,United States,NaN
8,Cancity,retail,2001,718.62,2448,United States,NaN
9,Cheers,entertainment,1993,4269.90,6472,United States,Massive Dynamic


In [10]:
# Step 3: Fixing the Issues

# 1. Fix the Date Trap
pipeline_df['engage_date'] = pd.to_datetime(pipeline_df['engage_date'], errors='coerce')
pipeline_df['close_date'] = pd.to_datetime(pipeline_df['close_date'], errors = 'coerce')

# 2. Impute the logical nulls
pipeline_df['close_value'] = pipeline_df['close_value'].fillna(0)
pipeline_df['account'] = pipeline_df['account'].fillna('Unknown')

# 3. Standardize account names and strip any accidental leading/trailing spaces just to be safe
accounts_df['account'] = accounts_df['account'].str.strip().str.title()

# 4: Fix the misspelled "technology" sector
accounts_df['sector'] = accounts_df['sector'].replace({'technolgy': 'technology'})

# verifying the fix by checking unique values in the sector column
print("Cleaned Sectors:", accounts_df['sector'].unique())

Cleaned Sectors: <StringArray>
[        'technology',            'medical',             'retail',
           'software',      'entertainment',          'marketing',
 'telecommunications',            'finance',         'employment',
           'services']
Length: 10, dtype: str


In [11]:
# Step 4: Merge the clean data
df_merged = pipeline_df.merge(teams_df, on='sales_agent', how='left')
df_merged = df_merged.merge(accounts_df, on='account', how='left')

print(f"Master Dataset Shape: {df_merged.shape}")

Master Dataset Shape: (8800, 16)


In [13]:
# Step 5: Feature Engineering

# 1. Calculate Sales Cycle Length
df_merged['sales_cycle_days'] = (df_merged['close_date'] - df_merged['engage_date']).dt.days

# 2. Aggregate Rep Performance
import numpy as np

rep_stats = df_merged.groupby('sales_agent').agg(
    total_deals=('opportunity_id', 'count'),
    won_deals=('deal_stage', lambda x: (x == 'Won').sum()),
    avg_deal_size=('close_value', lambda x: x[x > 0].mean())
).reset_index()

# 3. Handle any NaN values for reps with 0 wins
rep_stats['avg_deal_size'] = rep_stats['avg_deal_size'].fillna(0)

# 4. Calculate Win Rate (with a 1% floor to avoid dividing by zero)
rep_stats['win_rate'] = np.where(rep_stats['total_deals'] > 0, rep_stats['won_deals'] / rep_stats['total_deals'], 0.01)

# 5. The Rep Capacity Score: (Deal Volume * Avg Size) / Win Rate
rep_stats['capacity_score'] = (rep_stats['total_deals'] * rep_stats['avg_deal_size']) / rep_stats['win_rate']

# THE RESULTS
# Let's see the top 5 most overloaded reps
rep_stats.sort_values(by='capacity_score', ascending=False).head()

,sales_agent,total_deals,won_deals,avg_deal_size,win_rate,capacity_score
6,Darcel Schlecht,747,349,3304.338109,0.467202,5.283239e+06
2,Cassey Cress,346,163,2763.736196,0.471098,2.029837e+06
29,Zane Levy,349,161,2671.229814,0.461318,2.020860e+06
15,Kary Hendrixson,438,209,2173.674641,0.477169,1.995246e+06
26,Vicki Laflamme,451,221,2164.687783,0.490022,1.992306e+06


In [15]:
# Step 6: Export the cleaned master dataset
df_merged.to_csv('../dataset/processed/meridian_pipeline_clean.csv', index=False)
rep_stats.to_csv('../dataset/processed/meridian_rep_stats.csv', index=False)

print("Phase 1 Complete: Clean data exported to /data/processed/")

Phase 1 Complete: Clean data exported to /data/processed/
